# Original Boltz-2 1CLL TM-score benchmark (uv)

This notebook uses the public `boltz==2.2.1` command-line package in an isolated `uv` environment. It does not import the project's custom Boltz adapter or vendored Boltz source.

Start Jupyter from the project directory with:

Git Bash/Linux:

```bash
export JUPYTER_CONFIG_DIR="$PWD/.uv-cache/jupyter-config"
export JUPYTER_DATA_DIR="$PWD/.uv-cache/jupyter-data"
uv sync
uv run --with jupyterlab --with ipykernel --with stack-data jupyter lab
```

PowerShell:

```powershell
$env:JUPYTER_CONFIG_DIR = "$PWD/.uv-cache/jupyter-config"
$env:JUPYTER_DATA_DIR = "$PWD/.uv-cache/jupyter-data"
uv sync
uv run --with jupyterlab --with ipykernel --with stack-data jupyter lab
```

The generation cell repeats the original notebook's one-seed-per-process approach. It writes Boltz mmCIF files, converts them to PDB files, and then ranks them by TM-score.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter from inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
os.chdir(PROJECT_ROOT)
os.environ.setdefault("UV_CACHE_DIR", str(PROJECT_ROOT / ".uv-cache"))
os.environ.setdefault("BOLTZ_CACHE", str(PROJECT_ROOT / "data" / "boltz_cache"))
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv was not found on PATH. Launch Jupyter with the uv command shown above.")

if sys.platform == "win32":
    TORCH_SPEC = "torch==2.7.0+cu128"
    TORCH_INDEX = "https://download.pytorch.org/whl/cu128"
    RDKIT_SPEC = "rdkit>=2025.03.1"
else:
    TORCH_SPEC = "torch==2.5.1+cu118"
    TORCH_INDEX = "https://download.pytorch.org/whl/cu118"
    RDKIT_SPEC = "rdkit==2024.3.2"

ORIGINAL_BOLTZ_WITH = (
    "boltz==2.2.1",
    TORCH_SPEC,
    RDKIT_SPEC,
    "pandas==2.2.3",
    "pillow==10.4.0",
)


def original_boltz_run(*args: object, check: bool = True, capture_output: bool = False) -> subprocess.CompletedProcess[str]:
    """Run the public Boltz console command outside this project's dependencies."""
    command = [UV, "run", "--isolated", "--no-project"]
    for requirement in ORIGINAL_BOLTZ_WITH:
        command.extend(["--with", requirement])
    command.extend(["--index", TORCH_INDEX, "--index-strategy", "unsafe-best-match", "boltz", *[str(arg) for arg in args]])
    return subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        text=True,
        check=check,
        capture_output=capture_output,
    )


print("Project:", PROJECT_ROOT)
print("Notebook kernel:", sys.executable)
print("Public Boltz requirements:", ORIGINAL_BOLTZ_WITH)

Project: C:\Smile_lab\budgetting O3\Budgeting-O3
Notebook kernel: C:\Users\tnkch\AppData\Local\uv\cache\builds-v0\.tmpqzWcI0\Scripts\python.exe
Public Boltz requirements: ('boltz==2.2.1', 'torch==2.7.0+cu128', 'rdkit>=2025.03.1', 'pandas==2.2.3', 'pillow==10.4.0')


In [2]:
probe = original_boltz_run("--help", capture_output=True)
print(probe.stdout.splitlines()[0])
print("Public Boltz CLI resolved through uv.")

Usage: boltz [OPTIONS] COMMAND [ARGS]...
Public Boltz CLI resolved through uv.


In [3]:
import yaml

REFERENCE_PDB = PROJECT_ROOT / "data" / "1CLL.pdb"
INPUT_YAML = PROJECT_ROOT / "data" / "1cll_boltz_input.yaml"
CONFIG_YAML = PROJECT_ROOT / "configs" / "1cll.yaml"
if not REFERENCE_PDB.is_file():
    raise FileNotFoundError(REFERENCE_PDB)
if not INPUT_YAML.is_file():
    raise FileNotFoundError(INPUT_YAML)

input_data = yaml.safe_load(INPUT_YAML.read_text(encoding="utf-8"))
sequence = str(input_data["sequences"][0]["protein"]["sequence"]).strip().upper()
if len(sequence) != 144:
    raise ValueError(f"Expected 144 residues for 1CLL, got {len(sequence)}")

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "official_boltz_1cll_tm"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FASTA_PATH = OUTPUT_ROOT / "1CLL.fasta"
FASTA_PATH.write_text(f">A|protein\n{sequence}\n", encoding="utf-8")

print("Input FASTA:", FASTA_PATH)
print("Reference PDB:", REFERENCE_PDB)
print("Sequence length:", len(sequence))

Input FASTA: C:\Smile_lab\budgetting O3\Budgeting-O3\outputs\official_boltz_1cll_tm\1CLL.fasta
Reference PDB: C:\Smile_lab\budgetting O3\Budgeting-O3\data\1CLL.pdb
Sequence length: 144


## Generate original Boltz structures

Set `RUN_BENCHMARK = True` to run the 100 original `boltz predict` calls. Existing seed directories containing a CIF are skipped so an interrupted run can resume.

In [ ]:
N_SAMPLES = 100
SEED_START = 0
USE_MSA_SERVER = True
RUN_BENCHMARK = True

for seed in range(SEED_START, SEED_START + N_SAMPLES):
    seed_dir = OUTPUT_ROOT / f"seed_{seed:04d}"
    if list(seed_dir.rglob("*.cif")):
        print(f"Seed {seed} already completed")
        continue
    if not RUN_BENCHMARK:
        break

    print(f"\n===== Running original Boltz seed {seed} ({seed - SEED_START + 1}/{N_SAMPLES}) =====")
    boltz_args = [
        "predict",
        str(FASTA_PATH),
        "--out_dir",
        str(seed_dir),
        "--cache",
        str(PROJECT_ROOT / "data" / "boltz_cache"),
        "--seed",
        str(seed),
        "--no_kernels",
        "--output_format",
        "mmcif",
    ]
    if USE_MSA_SERVER:
        boltz_args.append("--use_msa_server")
    original_boltz_run(*boltz_args)

print("CIF files found:", len(list(OUTPUT_ROOT.rglob("*.cif"))))


===== Running original Boltz seed 0 (1/100) =====

===== Running original Boltz seed 1 (2/100) =====

===== Running original Boltz seed 2 (3/100) =====

===== Running original Boltz seed 3 (4/100) =====

===== Running original Boltz seed 4 (5/100) =====

===== Running original Boltz seed 5 (6/100) =====

===== Running original Boltz seed 6 (7/100) =====

===== Running original Boltz seed 7 (8/100) =====

===== Running original Boltz seed 8 (9/100) =====

===== Running original Boltz seed 9 (10/100) =====

===== Running original Boltz seed 10 (11/100) =====

===== Running original Boltz seed 11 (12/100) =====

===== Running original Boltz seed 12 (13/100) =====

===== Running original Boltz seed 13 (14/100) =====

===== Running original Boltz seed 14 (15/100) =====

===== Running original Boltz seed 15 (16/100) =====

===== Running original Boltz seed 16 (17/100) =====

===== Running original Boltz seed 17 (18/100) =====

===== Running original Boltz seed 18 (19/100) =====

===== Runni

In [ ]:
import gemmi
from tmtools import tm_align
from tmtools.io import get_residue_data, get_structure


def first_chain(structure, chain_id: str | None = None):
    model = next(structure.get_models())
    if chain_id is not None and chain_id in model:
        return model[chain_id]
    return next(model.get_chains())


cif_files = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*.cif")
    if "predictions" in path.parts
)
if not cif_files:
    raise FileNotFoundError("No Boltz CIF files found. Run the generation cell first.")

pdb_files = []
for cif_file in cif_files:
    pdb_file = cif_file.with_suffix(".pdb")
    if not pdb_file.is_file():
        gemmi.read_structure(str(cif_file)).write_pdb(str(pdb_file))
    pdb_files.append(pdb_file)

reference_structure = get_structure(str(REFERENCE_PDB))
reference_coords, reference_sequence = get_residue_data(first_chain(reference_structure, "A"))
results = []
for pdb_file in pdb_files:
    predicted_structure = get_structure(str(pdb_file))
    predicted_coords, predicted_sequence = get_residue_data(first_chain(predicted_structure))
    alignment = tm_align(
        predicted_coords,
        reference_coords,
        predicted_sequence,
        reference_sequence,
    )
    seed = next((part for part in pdb_file.parts if part.startswith("seed_")), "unknown")
    results.append({"seed": seed, "structure": str(pdb_file), "TM_score": float(alignment.tm_norm_chain1)})

df = pd.DataFrame(results).sort_values("TM_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1
print("CIF files converted to PDB:", len(pdb_files))
display(df.head(20))

In [ ]:
results_path = OUTPUT_ROOT / "1CLL_TMscore_results.csv"
df.to_csv(results_path, index=False)
print("Saved:", results_path)

best_k_df = pd.DataFrame({
    "K": range(1, len(df) + 1),
    "Best_TM_score": df["TM_score"].cummax().to_numpy(),
})
display(best_k_df[best_k_df["K"].isin([1, 5, 10, 20, 50, 100])])
print(f"Top-10 mean TM-score: {df.head(10)['TM_score'].mean():.4f}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed; skipping the optional plot.")
else:
    ax = best_k_df.plot("K", "Best_TM_score", figsize=(7, 5), legend=False)
    ax.set_title("Original Boltz-2 Best-of-K on 1CLL")
    ax.set_ylabel("Best TM-score")
    ax.grid(True)
    plt.show()